## prepare work

### Imports And Global Setup

这一格只放 notebook 全局依赖、dtype、随机种子和设备。

后面所有 cell 默认复用这里的全局变量：

- `device`
- `TRAIN_DTYPE`
- `THEORY_DTYPE`
- `THEORY_COMPLEX_DTYPE`
- `EPS_W`


In [ ]:
import math
import os
import time
from pathlib import Path
from typing import Callable, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
try:
    from torch.func import jacrev, vmap
    HAS_TORCH_FUNC = True
except Exception:
    jacrev = None
    vmap = None
    HAS_TORCH_FUNC = False

PAD_VALUE = 0.0
NP_DTYPE = np.float64
TRAIN_DTYPE = torch.float64
THEORY_DTYPE = torch.float64
THEORY_COMPLEX_DTYPE = torch.complex128
EPS_W = 1e-12
my_seed_number = 42

np.random.seed(my_seed_number)
torch.manual_seed(my_seed_number)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(my_seed_number)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"
cpu_budget = max(1, int(os.environ.get("SLURM_CPUS_PER_TASK", "1")))
torch.set_num_threads(cpu_budget)
torch.set_num_interop_threads(max(1, min(4, cpu_budget)))

print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Data And Training Configuration

- 定位训练数据
- 构造 `taus / mask / y`
- 对目标 `y=[delta, omega]` 做逐维标准化
- 准备训练集、验证集和 DataLoader
- 定义训练开关、checkpoint 路径和长度集合



In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATAPATH = PROJECT_ROOT / "data"
if not DATAPATH.exists():
    raise FileNotFoundError(f"DATAPATH does not exist: {DATAPATH}")

if "model_name" not in globals():
    raise RuntimeError("Please set model_name before running this notebook.")
if "retrain" not in globals():
    retrain = False
if "load_training_data" not in globals():
    load_training_data = True

default_data_root = DATAPATH
runtime_data_root = Path(os.environ.get("PARAMEST_DATA_ROOT", str(default_data_root))).resolve()
if not runtime_data_root.exists():
    raise FileNotFoundError(f"runtime data root not found: {runtime_data_root}")

dataset_root = runtime_data_root / "tragectories" / model_name
path_tau = dataset_root / "trajectories.npy"
path_param = dataset_root / "params.npy"

L_max = 50
target_ntraj = 1_000_000
default_batch_size = 131072 if device.type == "cuda" else 1024
batch_size = int(globals().get("batch_size", default_batch_size))
num_epochs = 60 if device.type == "cuda" else 4
learning_rate = 2e-3
weight_decay = 1e-4
train_fraction = 0.8
train_length_choices = [10, 15, 20, 30, 40, 50]
alpha_init_range = (0.1, 1.5)
beta_init_range = (-5.0, 5.0)
loader_workers = max(0, min(16, cpu_budget - 1))
loader_prefetch = int(globals().get("loader_prefetch_factor", 4))
log_every = int(globals().get("log_every", 5 if device.type == "cuda" else 1))
save_checkpoint = bool(globals().get("save_checkpoint", True))
use_amp = False
amp_dtype_name = "disabled"
amp_dtype = None
loader_kwargs = {
    "num_workers": loader_workers,
    "pin_memory": pin_memory,
}
if loader_workers > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = max(2, loader_prefetch)

checkpoint_dir = default_data_root / "models" / model_name
checkpoint_dir.mkdir(parents=True, exist_ok=True)
model_ckpt_path = checkpoint_dir / f"pole_residue_block_head_{model_name}.pt"
ckpt_for_load = torch.load(model_ckpt_path, map_location="cpu") if model_ckpt_path.exists() else None

def paramblocks(y_raw):
    y_raw = np.asarray(y_raw, dtype=NP_DTYPE)
    params = y_raw[:, :2]
    if len(params) == 0:
        return np.zeros((0, 2), dtype=np.int64)
    changes = np.any(params[1:] != params[:-1], axis=1)
    starts = np.r_[0, np.flatnonzero(changes) + 1]
    stops = np.r_[starts[1:], len(params)]
    return np.stack([starts, stops], axis=1).astype(np.int64, copy=False)

def splitblocks(taus_raw, y_raw, train_frac=0.8, seed=42):
    blocks = paramblocks(y_raw)
    if len(blocks) < 2:
        raise ValueError("Need at least two parameter blocks for train/val split.")

    rng = np.random.default_rng(seed)
    order = rng.permutation(len(blocks))
    n_train = int(round(train_frac * len(blocks)))
    n_train = min(max(n_train, 1), len(blocks) - 1)

    train_blocks = blocks[order[:n_train]]
    val_blocks = blocks[order[n_train:]]

    def gather(arr, chosen_blocks):
        parts = [np.asarray(arr[start:stop]) for start, stop in chosen_blocks]
        return np.concatenate(parts, axis=0)

    x_train = gather(taus_raw, train_blocks)
    y_train = gather(y_raw, train_blocks)
    x_val = gather(taus_raw, val_blocks)
    y_val = gather(y_raw, val_blocks)

    info = {
        "train_blocks": int(len(train_blocks)),
        "val_blocks": int(len(val_blocks)),
        "train_rows": int(len(x_train)),
        "val_rows": int(len(x_val)),
    }
    return x_train, x_val, y_train, y_val, info

def _as_laplace_dataset(taus_raw, y_raw):
    taus_raw = np.asarray(taus_raw, dtype=NP_DTYPE)
    y_raw = np.asarray(y_raw, dtype=NP_DTYPE)
    valid_mask = np.isfinite(taus_raw) & (taus_raw > 0.0)
    lengths = np.sum(valid_mask, axis=1).astype(np.int64)
    if np.any(lengths == 0):
        raise ValueError("Found trajectories with zero valid waiting times after preprocessing.")

    taus = np.where(valid_mask, taus_raw, PAD_VALUE).astype(NP_DTYPE, copy=False)
    y = y_raw[:, :2].astype(NP_DTYPE, copy=False)

    return {
        "taus": torch.from_numpy(taus),
        "mask": torch.from_numpy(valid_mask.astype(np.bool_, copy=False)),
        "lengths": torch.from_numpy(lengths),
        "y": torch.from_numpy(y),
    }

if load_training_data:
    if not path_tau.exists():
        raise FileNotFoundError(f"trajectory file not found: {path_tau}")
    if not path_param.exists():
        raise FileNotFoundError(f"param file not found: {path_param}")

    tau_all = np.load(path_tau, mmap_mode="r")
    param_all = np.load(path_param, mmap_mode="r")
    n_total = min(tau_all.shape[0], param_all.shape[0])
    ntraj_select = min(target_ntraj, n_total)

    if ntraj_select < n_total:
        rng = np.random.default_rng(my_seed_number)
        select_idx = np.sort(rng.choice(n_total, size=ntraj_select, replace=False))
    else:
        select_idx = np.arange(n_total)

    tau_list = np.asarray(tau_all[select_idx, :L_max], dtype=NP_DTYPE)
    param_list = np.asarray(param_all[select_idx], dtype=NP_DTYPE)
    print(f"Loaded {ntraj_select:,}/{n_total:,} trajectories from {path_tau}")

    x_train, x_valid, y_train, y_valid, split_info = splitblocks(
        tau_list,
        param_list,
        train_frac=train_fraction,
        seed=my_seed_number,
    )

    train_data = _as_laplace_dataset(x_train, y_train)
    val_data = _as_laplace_dataset(x_valid, y_valid)
    target_mean = train_data["y"].mean(dim=0).to(dtype=TRAIN_DTYPE)
    target_std = train_data["y"].std(dim=0, unbiased=False).clamp_min(1e-6).to(dtype=TRAIN_DTYPE)

    train_loader = DataLoader(
        TensorDataset(train_data["taus"], train_data["mask"], train_data["y"]),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        **loader_kwargs,
    )
    val_loader = DataLoader(
        TensorDataset(val_data["taus"], val_data["mask"], val_data["y"]),
        batch_size=batch_size,
        shuffle=False,
        **loader_kwargs,
    )
    print(f"model_name={model_name}")
    print(f"train_data: {tuple(train_data['taus'].shape)}, val_data: {tuple(val_data['taus'].shape)}")
    print(f"split blocks: train={split_info['train_blocks']}, val={split_info['val_blocks']}")
    print(f"split rows: train={split_info['train_rows']:,}, val={split_info['val_rows']:,}")
    print(f"runtime_data_root={runtime_data_root}")
    print(f"cpu_budget={cpu_budget}, loader_workers={loader_workers}, loader_prefetch={loader_prefetch}, use_amp={use_amp}, amp_dtype={amp_dtype_name}")
    print(f"log_every={log_every}, save_checkpoint={save_checkpoint}")
else:
    if ckpt_for_load is None:
        raise FileNotFoundError(f"checkpoint not found: {model_ckpt_path}")
    state = ckpt_for_load["model_state"]
    target_mean = state["target_mean"].to(dtype=TRAIN_DTYPE)
    target_std = state["target_std"].to(dtype=TRAIN_DTYPE)
    train_data = None
    val_data = None
    train_loader = None
    val_loader = None
    split_info = None
    print(f"model_name={model_name}")
    print(f"Loaded checkpoint only, skipped dataset load: {model_ckpt_path}")

print(f"batch_size={batch_size}, num_epochs={num_epochs}, train_length_choices={train_length_choices}")
print(f"alpha_init_range={alpha_init_range}, beta_init_range={beta_init_range}")
print("target_mean =", target_mean)
print("target_std  =", target_std)


## Model Definition

1. 基础 complex Laplace 特征层
2. 读出几何所需的局部 helper
3. 最终估计器类和训练工具

### Complex Laplace Feature Layer

输入是 padded waiting-time 序列和对应 `mask`。

输出是：

- 所有 probe 的实部平均
- 所有 probe 的虚部平均
- 一个固定长度特征 `1/sqrt(N)`

这里学到的是每个 probe 的 `alpha_k` 和 `beta_k`。


In [ ]:
def _init_probe_values(init, K: int, dtype: torch.dtype) -> torch.Tensor:
    init_t = torch.as_tensor(init, dtype=dtype)
    if init_t.ndim == 1 and init_t.numel() == K:
        return init_t.clone()
    if init_t.ndim == 1 and init_t.numel() == 2:
        return torch.linspace(init_t[0], init_t[1], K, dtype=dtype)
    raise ValueError(f"Expected probe initializer with shape ({K},) or (2,), got {tuple(init_t.shape)}")


class ComplexLaplaceFeatures(nn.Module):
    """
    Empirical complex Laplace probes:
        F_k = (1/N) * sum_n exp(-(alpha_k - i beta_k) * tau_n)

    Output features:
        [Re(F_1), ..., Re(F_K), Im(F_1), ..., Im(F_K), 1/sqrt(N)]
    """

    def __init__(
        self,
        K: int = 3,
        alpha_init=(0.2, 0.8),
        beta_init=(-2.0, 2.0),
        eps_alpha: float = 1e-4,
    ):
        super().__init__()
        self.K = K
        self.eps_alpha = eps_alpha

        alpha0 = _init_probe_values(alpha_init, K, TRAIN_DTYPE)
        beta0 = _init_probe_values(beta_init, K, TRAIN_DTYPE)

        raw_alpha0 = torch.log(torch.expm1(torch.clamp(alpha0 - eps_alpha, min=1e-6)))
        self.raw_alpha = nn.Parameter(raw_alpha0)
        self.beta = nn.Parameter(beta0)
        self.output_dim = 2 * K + 1

    def forward(self, taus: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        taus = taus.to(dtype=self.raw_alpha.dtype)
        mask = mask.to(dtype=self.raw_alpha.dtype)

        N = mask.sum(dim=1, keepdim=True).clamp_min(1.0)
        alpha = F.softplus(self.raw_alpha) + self.eps_alpha
        beta = self.beta

        tau = taus.unsqueeze(-1)
        m = mask.unsqueeze(-1)

        real_part = torch.exp(-tau * alpha) * torch.cos(tau * beta)
        imag_part = torch.exp(-tau * alpha) * torch.sin(tau * beta)

        re = (real_part * m).sum(dim=1) / N
        im = (imag_part * m).sum(dim=1) / N
        lf = torch.rsqrt(N)

        return torch.cat([re, im, lf], dim=1)


### Readout Geometry Helpers

- Model Laplace 头部公式
- `delta` 的局部切向方向

它们会被后面的 block-structured 读出类复用。


In [ ]:
def modelhead(theta, s, eps=EPS_W):
    if "laplace_transform_torch" not in globals():
        raise RuntimeError("Define laplace_transform_torch(theta, s, eps=...) before running.")

    theta = theta.to(dtype=TRAIN_DTYPE)
    s = s.to(dtype=THEORY_COMPLEX_DTYPE)
    squeeze = theta.ndim == 1
    if squeeze:
        return laplace_transform_torch(theta, s, eps=eps)

    try:
        out = laplace_transform_torch(theta, s, eps=eps)
        if out.shape[0] == theta.shape[0]:
            return out
    except Exception:
        pass

    if HAS_TORCH_FUNC:
        return vmap(lambda th: laplace_transform_torch(th, s, eps=eps))(theta)
    outs = [laplace_transform_torch(theta_i, s, eps=eps) for theta_i in theta]
    return torch.stack(outs, dim=0)


def _delta_tangent_autograd(alpha, beta, theta_ref):
    s = torch.complex(alpha, -beta)

    def _mu_parts(theta_local):
        mu = modelhead(theta_local, s)
        return torch.stack([torch.real(mu), torch.imag(mu)], dim=1)

    if HAS_TORCH_FUNC:
        jac = vmap(jacrev(_mu_parts))(theta_ref)
        d_re_d_delta = jac[:, :, 0, 0]
        d_im_d_delta = jac[:, :, 1, 0]
        out = torch.stack([d_re_d_delta, d_im_d_delta], dim=2).to(dtype=alpha.dtype)
        return out.detach()

    th_req = theta_ref.detach().clone().requires_grad_(True)

    def _mu_batch(th_batch):
        mu = modelhead(th_batch, s)
        return torch.stack([torch.real(mu), torch.imag(mu)], dim=2)

    jac = torch.autograd.functional.jacobian(
        _mu_batch,
        th_req,
        create_graph=False,
        strict=False,
        vectorize=True,
    )
    batch_idx = torch.arange(theta_ref.shape[0], device=theta_ref.device)
    jac_diag = jac[batch_idx, :, :, batch_idx, :]
    d_re_d_delta = jac_diag[:, :, 0, 0]
    d_im_d_delta = jac_diag[:, :, 1, 0]
    return torch.stack([d_re_d_delta, d_im_d_delta], dim=2).to(dtype=alpha.dtype)


def dtangent(alpha, beta, theta_ref):
    squeeze = theta_ref.ndim == 1
    if squeeze:
        theta_ref = theta_ref.unsqueeze(0)

    theta_ref = theta_ref.to(device=alpha.device, dtype=alpha.dtype)
    g = _delta_tangent_autograd(alpha, beta, theta_ref)
    if not torch.isfinite(g).all():
        raise RuntimeError("Non-finite tangent values encountered in autograd path.")
    return g[0] if squeeze else g


### Block-Structured Estimator

- `omega`：每个 probe 平面里学一个方向，再做稳定混合
- `delta`：在每个 probe 平面里取去掉 `omega` 分量后的局部方向

最终理论分析仍然通过 `linearparams()` 抽成等效线性 `W, c`。


In [ ]:
class LaplaceLinearEstimator(nn.Module):
    """
    Complex Laplace feature layer + structured block heads, without whitening.
    omega uses free signed coefficients over learned per-probe 2D directions.
    delta uses per-probe directions obtained from the exact local tangent at the current point.
    """

    def __init__(
        self,
        out_dim: int = 2,
        K: int = 3,
        alpha_init=(0.2, 0.8),
        beta_init=(-2.0, 2.0),
        target_mean: torch.Tensor = None,
        target_std: torch.Tensor = None,
        omega_init_scale: float = 0.1,
        delta_steps: int = 4,
    ):
        super().__init__()
        if target_mean is None or target_std is None:
            raise ValueError("target_mean and target_std must be provided explicitly.")
        if out_dim != 2:
            raise ValueError("This notebook expects out_dim=2 for [delta, omega].")

        self.feat = ComplexLaplaceFeatures(
            K=K,
            alpha_init=alpha_init,
            beta_init=beta_init,
        )
        self.in_dim = self.feat.output_dim
        self.K = int(K)
        self.delta_steps = int(delta_steps)

        self.delta_coeff = nn.Parameter(torch.zeros(self.K, dtype=TRAIN_DTYPE))
        self.delta_length_weight = nn.Parameter(torch.zeros((), dtype=TRAIN_DTYPE))
        self.delta_bias = nn.Parameter(torch.zeros((), dtype=TRAIN_DTYPE))

        self.omega_dir_raw = nn.Parameter(torch.randn(self.K, 2, dtype=TRAIN_DTYPE) * omega_init_scale)
        self.omega_coeff = nn.Parameter(torch.zeros(self.K, dtype=TRAIN_DTYPE))
        self.omega_length_weight = nn.Parameter(torch.zeros((), dtype=TRAIN_DTYPE))
        self.omega_bias = nn.Parameter(torch.zeros((), dtype=TRAIN_DTYPE))

        self.register_buffer("head_theta_ref", torch.as_tensor(target_mean, dtype=TRAIN_DTYPE))
        self.register_buffer("target_mean", torch.as_tensor(target_mean, dtype=TRAIN_DTYPE))
        self.register_buffer(
            "target_std",
            torch.clamp(torch.as_tensor(target_std, dtype=TRAIN_DTYPE), min=1e-6),
        )

    def omegadirs(self) -> torch.Tensor:
        return self.omega_dir_raw / torch.clamp(torch.linalg.norm(self.omega_dir_raw, dim=1, keepdim=True), min=1e-6)

    def omegaweights(self) -> torch.Tensor:
        dirs = self.omegadirs()
        return self.omega_coeff.unsqueeze(1) * dirs

    def deltadirs(self, theta_ref: torch.Tensor) -> torch.Tensor:
        alpha = F.softplus(self.feat.raw_alpha) + self.feat.eps_alpha
        beta = self.feat.beta
        q = self.omegadirs()
        g = dtangent(alpha, beta, theta_ref)
        if g.ndim == 2:
            proj = g - torch.sum(g * q, dim=1, keepdim=True) * q
            proj_norm = torch.linalg.norm(proj, dim=1, keepdim=True)
            fallback = torch.stack([-q[:, 1], q[:, 0]], dim=1)
            return torch.where(proj_norm > 1e-6, proj / proj_norm.clamp_min(1e-6), fallback)

        q = q.unsqueeze(0)
        proj = g - torch.sum(g * q, dim=2, keepdim=True) * q
        proj_norm = torch.linalg.norm(proj, dim=2, keepdim=True)
        fallback = torch.stack([-q[..., 1], q[..., 0]], dim=2)
        return torch.where(proj_norm > 1e-6, proj / proj_norm.clamp_min(1e-6), fallback)

    def splitx(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return x[:, : self.K], x[:, self.K : 2 * self.K], x[:, -1]

    def omegastd(self, x: torch.Tensor) -> torch.Tensor:
        xre, xim, xlen = self.splitx(x)
        omega_probe = self.omegaweights()
        return xre @ omega_probe[:, 0] + xim @ omega_probe[:, 1] + xlen * self.omega_length_weight + self.omega_bias

    def deltastd(self, x: torch.Tensor, omega_std: torch.Tensor) -> torch.Tensor:
        xre, xim, xlen = self.splitx(x)
        omega = self.decode_targets(torch.stack([torch.zeros_like(omega_std), omega_std], dim=1))[:, 1]
        delta = self.target_mean[0].expand_as(omega)

        for _ in range(self.delta_steps):
            theta_ref = torch.stack([delta, omega], dim=1)
            dirs = self.deltadirs(theta_ref)
            delta_std = torch.sum(xre * (self.delta_coeff.unsqueeze(0) * dirs[:, :, 0]), dim=1)
            delta_std = delta_std + torch.sum(xim * (self.delta_coeff.unsqueeze(0) * dirs[:, :, 1]), dim=1)
            delta_std = delta_std + xlen * self.delta_length_weight + self.delta_bias
            delta = self.decode_targets(torch.stack([delta_std, omega_std], dim=1))[:, 0]

        return delta_std

    def linearparams(self, theta_ref: torch.Tensor = None):
        if theta_ref is None:
            theta_ref = self.head_theta_ref
        if theta_ref.ndim != 1:
            raise ValueError("linearparams expects a single reference point.")
        W = torch.zeros((2, self.in_dim), dtype=self.delta_coeff.dtype, device=self.delta_coeff.device)

        delta_dirs = self.deltadirs(theta_ref)
        W[0, : self.K] = self.delta_coeff * delta_dirs[:, 0]
        W[0, self.K : 2 * self.K] = self.delta_coeff * delta_dirs[:, 1]
        W[0, -1] = self.delta_length_weight

        omega_probe = self.omegaweights()
        W[1, : self.K] = omega_probe[:, 0]
        W[1, self.K : 2 * self.K] = omega_probe[:, 1]
        W[1, -1] = self.omega_length_weight

        c = torch.stack([self.delta_bias, self.omega_bias])
        return W, c

    def encode_targets(self, y: torch.Tensor) -> torch.Tensor:
        y = y.to(device=self.target_mean.device, dtype=self.target_mean.dtype)
        return (y - self.target_mean) / self.target_std

    def decode_targets(self, y_std: torch.Tensor) -> torch.Tensor:
        y_std = y_std.to(device=self.target_mean.device, dtype=self.target_mean.dtype)
        return y_std * self.target_std + self.target_mean

    def forward_standardized(self, taus: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        x = self.feat(taus, mask)
        omega_std = self.omegastd(x)
        delta_std = self.deltastd(x, omega_std)
        return torch.stack([delta_std, omega_std], dim=1)

    def forward(self, taus: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        return self.decode_targets(self.forward_standardized(taus, mask))

    @torch.no_grad()
    def probe_parameters(self, detach: bool = True) -> Tuple[torch.Tensor, torch.Tensor]:
        alpha = F.softplus(self.feat.raw_alpha) + self.feat.eps_alpha
        beta = self.feat.beta
        if detach:
            alpha = alpha.detach()
            beta = beta.detach()
        return alpha, beta


def adaptstate(state_dict):
    upgraded = {
        k: (v.clone() if isinstance(v, torch.Tensor) else v)
        for k, v in state_dict.items()
    }

    legacy_omega = (
        "omega_coeff" not in upgraded
        and "omega_gate_logits" in upgraded
        and "omega_scale_raw" in upgraded
    )
    if legacy_omega:
        gates = torch.softmax(upgraded["omega_gate_logits"].to(dtype=TRAIN_DTYPE), dim=0)
        scale = F.softplus(upgraded["omega_scale_raw"].to(dtype=TRAIN_DTYPE))
        upgraded["omega_coeff"] = scale * gates

    upgraded.pop("omega_gate_logits", None)
    upgraded.pop("omega_scale_raw", None)
    return upgraded, legacy_omega




### Training Utilities

这里放训练环节真正会用到的小工具：

- 损失函数
- 随机截断长度
- 验证接口
- 主训练循环

把这些和模型类拆开以后，训练部分更容易单独定位。


In [ ]:
def mse_loss(model: LaplaceLinearEstimator, pred_std: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return F.mse_loss(pred_std, model.encode_targets(target))


def truncbatch(taus: torch.Tensor, mask: torch.Tensor, length: int) -> Tuple[torch.Tensor, torch.Tensor]:
    L = int(length)
    return taus[:, :L], mask[:, :L]


def randtrunc(
    taus: torch.Tensor,
    mask: torch.Tensor,
    length_choices,
) -> Tuple[torch.Tensor, torch.Tensor, int]:
    valid_choices = [int(L) for L in length_choices if int(L) > 0]
    if not valid_choices:
        raise ValueError("length_choices must contain at least one positive integer.")
    choice_idx = torch.randint(len(valid_choices), (1,), device=taus.device).item()
    chosen_length = valid_choices[choice_idx]
    trunc_taus, trunc_mask = truncbatch(taus, mask, chosen_length)
    return trunc_taus, trunc_mask, chosen_length


def evalmodel(model, loader, device, length=None):
    model.eval()
    total_loss = 0.0
    total_count = 0
    total_sqerr = torch.zeros(model.target_mean.shape[0], dtype=TRAIN_DTYPE, device=device)
    non_blocking = device.type == "cuda"

    with torch.no_grad():
        for taus, mask, y in loader:
            taus = taus.to(device, non_blocking=non_blocking)
            mask = mask.to(device, non_blocking=non_blocking)
            y = y.to(device, non_blocking=non_blocking)

            if length is not None:
                taus, mask = truncbatch(taus, mask, int(length))

            pred_std = model.forward_standardized(taus, mask)
            pred = model.decode_targets(pred_std)
            data_loss = mse_loss(model, pred_std, y)
            batch_n = y.shape[0]
            total_loss += data_loss.item() * batch_n
            total_count += batch_n
            total_sqerr += torch.sum((pred - y) ** 2, dim=0)

    rmse = torch.sqrt(total_sqerr / max(total_count, 1))
    return {
        "std_mse": total_loss / max(total_count, 1),
        "rmse": rmse,
    }


def evallengths(model, loader, device, lengths):
    by_length = {}
    scores = []
    for L in [int(L) for L in lengths]:
        metrics = evalmodel(model, loader, device, length=L)
        by_length[L] = metrics
        scores.append(metrics["std_mse"])
    return {
        "avg_std_mse": float(np.mean(scores)),
        "by_length": by_length,
    }


def bandsummary(model, data, device, length, batch_size=4096):
    y = data["y"]
    delta = y[:, 0]
    omega = y[:, 1]

    d_edges = torch.linspace(float(delta.min()), float(delta.max()), 4)
    o_edges = torch.linspace(float(omega.min()), float(omega.max()), 4)
    specs = {
        "delta_low": (delta >= d_edges[0]) & (delta < d_edges[1]),
        "delta_mid": (delta >= d_edges[1]) & (delta < d_edges[2]),
        "delta_high": delta >= d_edges[2],
        "omega_low": (omega >= o_edges[0]) & (omega < o_edges[1]),
        "omega_mid": (omega >= o_edges[1]) & (omega < o_edges[2]),
        "omega_high": omega >= o_edges[2],
    }

    summary = {}
    for name, keep in specs.items():
        count = int(keep.sum().item())
        if count == 0:
            continue
        subset = TensorDataset(data["taus"][keep], data["mask"][keep], data["y"][keep])
        loader = DataLoader(
            subset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=min(2, loader_workers),
            pin_memory=pin_memory,
        )
        metrics = evalmodel(model, loader, device, length=length)
        summary[name] = {
            "count": count,
            "rmse": metrics["rmse"].detach().cpu().tolist(),
        }
    return summary


def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    epochs,
    train_length_choices,
):
    history = {"train_std_mse": [], "val_std_mse": [], "val_rmse": [], "val_by_length": [], "epoch_timing": []}
    best_val = float("inf")
    best_state = None
    best_val_rmse = None
    best_val_by_length = None
    non_blocking = device.type == "cuda"

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        running_count = 0
        running_sqerr = torch.zeros(model.target_mean.shape[0], dtype=TRAIN_DTYPE, device=device)
        chosen_lengths = []

        data_time = 0.0
        compute_time = 0.0
        tick = time.perf_counter()
        for taus, mask, y in train_loader:
            t_data_ready = time.perf_counter()
            data_time += t_data_ready - tick
            taus = taus.to(device, non_blocking=non_blocking)
            mask = mask.to(device, non_blocking=non_blocking)
            y = y.to(device, non_blocking=non_blocking)

            taus, mask, chosen_length = randtrunc(taus, mask, train_length_choices)
            chosen_lengths.append(chosen_length)

            optimizer.zero_grad(set_to_none=True)
            t_compute_start = time.perf_counter()
            pred_std = model.forward_standardized(taus, mask)
            pred = model.decode_targets(pred_std)
            data_loss = mse_loss(model, pred_std, y)
            data_loss.backward()
            optimizer.step()
            compute_time += time.perf_counter() - t_compute_start
            tick = time.perf_counter()

            batch_n = y.shape[0]
            running_loss += data_loss.item() * batch_n
            running_count += batch_n
            running_sqerr += torch.sum((pred.detach() - y) ** 2, dim=0)

        train_std_mse = running_loss / max(running_count, 1)
        train_rmse = torch.sqrt(running_sqerr / max(running_count, 1))
        val_summary = evallengths(model, val_loader, device, train_length_choices)
        max_length = max(int(L) for L in train_length_choices)
        val_metrics = val_summary["by_length"][max_length]
        history["train_std_mse"].append(train_std_mse)
        history["val_std_mse"].append(val_summary["avg_std_mse"])
        history["val_rmse"].append(val_metrics["rmse"].detach().cpu().tolist())
        history["val_by_length"].append({
            int(L): val_summary["by_length"][int(L)]["rmse"].detach().cpu().tolist()
            for L in train_length_choices
        })

        if val_summary["avg_std_mse"] < best_val:
            best_val = val_summary["avg_std_mse"]
            best_val_rmse = val_metrics["rmse"].detach().cpu()
            best_val_by_length = history["val_by_length"][-1]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        used_lengths = sorted(set(chosen_lengths))
        history["epoch_timing"].append({"data_time": data_time, "compute_time": compute_time})
        if (epoch + 1) % max(1, log_every) == 0 or (epoch + 1) == epochs:
            print(
                f"epoch {epoch + 1:02d}/{epochs} "
                f"train_std_mse={train_std_mse:.6f} "
                f"train_rmse={train_rmse.detach().cpu().numpy()} "
                f"val_std_mse_avg={val_summary['avg_std_mse']:.6f} "
                f"val_rmse_L{max_length}={val_metrics['rmse'].detach().cpu().numpy()} "
                f"used_lengths={used_lengths} "
                f"data_time={data_time:.2f}s compute_time={compute_time:.2f}s"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    history["best_val_std_mse"] = best_val
    history["best_val_rmse"] = best_val_rmse
    history["best_val_by_length"] = best_val_by_length
    return history


## Training And Loading


In [ ]:
if (not retrain) and (ckpt_for_load is not None) and ("K_probes" in ckpt_for_load) and ("K_probes" not in globals()):
    K_probes = int(ckpt_for_load["K_probes"])
else:
    K_probes = int(globals().get("K_probes", 4))

model = LaplaceLinearEstimator(
    out_dim=2,
    K=K_probes,
    alpha_init=alpha_init_range,
    beta_init=beta_init_range,
    target_mean=target_mean,
    target_std=target_std,
).to(device)

print("beta_init range =", beta_init_range)
print("feature_dim =", model.in_dim)
print("retrain =", retrain)
print("checkpoint =", model_ckpt_path)

did_train = False
if retrain or not model_ckpt_path.exists():
    if train_loader is None or val_loader is None:
        raise RuntimeError("Training requires load_training_data=True.")
    if not retrain and not model_ckpt_path.exists():
        print(f"Checkpoint not found, training from scratch: {model_ckpt_path}")
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    history = train_model(
        model,
        train_loader,
        val_loader,
        optimizer,
        device=device,
        epochs=num_epochs,
        train_length_choices=train_length_choices,
    )
    did_train = True
else:
    ckpt = ckpt_for_load if ckpt_for_load is not None else torch.load(model_ckpt_path, map_location=device)
    model_state, adapted = adaptstate(ckpt["model_state"])
    model.load_state_dict(model_state)
    history = ckpt.get("history", None)
    if adapted:
        print("Loaded legacy omega head checkpoint and converted it to free omega coefficients.")
    print(f"Loaded model checkpoint from {model_ckpt_path}")

final_val = None
final_val_by_length = None
final_val_bands = None
if val_loader is not None and val_data is not None:
    max_length = max(int(L) for L in train_length_choices)
    final_val = evallengths(model, val_loader, device, train_length_choices)
    final_val_by_length = {
        int(L): final_val["by_length"][int(L)]["rmse"].detach().cpu().tolist()
        for L in train_length_choices
    }
    final_val_bands = bandsummary(
        model,
        val_data,
        device=device,
        length=max_length,
        batch_size=batch_size,
    )
    if history is not None:
        history["final_val_by_length"] = final_val_by_length
        history["final_val_bands"] = final_val_bands

alpha, beta = model.probe_parameters()
W_eff, c_eff = model.linearparams()
if history is not None and "best_val_std_mse" in history:
    print(f"Best validation standardized MSE: {history['best_val_std_mse']:.6f}")
if history is not None and "best_val_rmse" in history and history["best_val_rmse"] is not None:
    best_rmse = history["best_val_rmse"]
    if torch.is_tensor(best_rmse):
        best_rmse = best_rmse.detach().cpu().numpy()
    print("Best validation RMSE [delta, omega]:", best_rmse)
if final_val is not None:
    print(f"Validation standardized MSE averaged over lengths: {final_val['avg_std_mse']:.6f}")
    for L in sorted(final_val_by_length):
        print(f"Validation RMSE L={L}: {np.array(final_val_by_length[L])}")
    print("Validation RMSE by band:")
    for name, stats in final_val_bands.items():
        print(f"  {name}: count={stats['count']}, rmse={np.array(stats['rmse'])}")
print("Learned alpha:", alpha.detach().cpu().numpy())
print("Learned beta :", beta.detach().cpu().numpy())
print("Effective standardized weight rows at target_mean:")
print(W_eff.detach().cpu())
print("Effective standardized bias:", c_eff.detach().cpu())


### Save Current Model

这一格把当前模型存到 checkpoint 文件里。

会一起保存：

- `model_state`
- `history`
- 关键训练超参数


In [ ]:
if did_train and save_checkpoint:
    ckpt_payload = {
        "model_state": model.state_dict(),
        "history": history,
        "K_probes": K_probes,
        "train_length_choices": train_length_choices,
        "alpha_init_range": alpha_init_range,
        "beta_init_range": beta_init_range,
    }
    torch.save(ckpt_payload, model_ckpt_path)
    print(f"Saved checkpoint to {model_ckpt_path}")
elif did_train and not save_checkpoint:
    print("Skipped checkpoint save because save_checkpoint=False.")
else:
    print("Skipped checkpoint save because the model was only loaded for analysis.")


## Theory: Exact MSE, biased-CRB, And Barankin

- 特征矩
- exact MSE
- biased-CRB
- generalized Barankin bound


### Base Theory Helpers

- tensor / complex tensor 统一入口
- 数值积分
- Model waiting-time pdf
- Model Laplace 变换
- 从训练好模型中抽取等效线性参数
- 数值 Laplace 验证接口


#### Tensor, Model Model, And Laplace Utilities

这一格包含理论部分最底层的工具函数。

- `s` 的定义
- Model pdf
- Model Laplace 闭式
- 等效线性参数如何从模型中读出



In [ ]:
def as_real_tensor(x, *, device=device, dtype=THEORY_DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(device=device, dtype=dtype)
    return torch.as_tensor(x, device=device, dtype=dtype)


def as_complex_tensor(x, *, device=device, dtype=THEORY_COMPLEX_DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(device=device, dtype=dtype)
    return torch.as_tensor(x, device=device, dtype=dtype)


def tensor_to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def trapz_torch(y, x):
    if y.shape[0] < 2:
        return torch.zeros(y.shape[1:], dtype=y.dtype, device=y.device)
    dx = (x[1:] - x[:-1]).to(dtype=y.dtype)
    if y.ndim > 1:
        dx = dx.view(dx.shape[0], *([1] * (y.ndim - 1)))
    return torch.sum(0.5 * (y[1:] + y[:-1]) * dx, dim=0)


def length_feature_value(N_obs, device=device):
    N_obs = as_real_tensor(N_obs, device=device)
    return torch.rsqrt(N_obs)


if "waiting_time_real_torch" not in globals():
    def waiting_time_real_torch(tau, delta, omega, eps=EPS_W):
        raise RuntimeError(
            "Define waiting_time_real_torch(tau, delta, omega, eps=...) before running theory cells."
        )


if "laplace_transform_torch" not in globals():
    def laplace_transform_torch(theta, s, eps=EPS_W):
        raise RuntimeError(
            "Define laplace_transform_torch(theta, s, eps=...) before running theory cells."
        )




#### Feature Moments And Fisher Information

这一格把 Laplace probe 的一阶、二阶矩整理成最终特征均值和协方差，并给出 fixed-`N` Fisher 信息。

exact MSE 和 biased-CRB 的主要输入都来自这里。


In [ ]:
def complex_to_real_probe_map(K, device):
    T = torch.zeros((2 * K, 2 * K), dtype=THEORY_COMPLEX_DTYPE, device=device)
    idx = torch.arange(K, device=device)
    T[idx, idx] = 0.5
    T[idx, K + idx] = 0.5
    T[K + idx, idx] = -0.5j
    T[K + idx, K + idx] = 0.5j
    return T


def one_sample_complex_probe_moments(theta, s, laplace_w):
    theta = as_real_tensor(theta)
    s = as_complex_tensor(s, device=theta.device)
    aug_s = torch.cat([s, torch.conj(s)], dim=0)
    m1 = laplace_w(theta, aug_s)
    m2 = laplace_w(theta, aug_s[:, None] + aug_s[None, :])
    m3 = laplace_w(theta, aug_s[:, None, None] + aug_s[None, :, None] + aug_s[None, None, :])
    m4 = laplace_w(
        theta,
        aug_s[:, None, None, None]
        + aug_s[None, :, None, None]
        + aug_s[None, None, :, None]
        + aug_s[None, None, None, :],
    )
    return m1, m2, m3, m4


def one_sample_real_probe_moments(theta, s, laplace_w):
    theta = as_real_tensor(theta)
    s = as_complex_tensor(s, device=theta.device)
    K = int(s.numel())
    T = complex_to_real_probe_map(K, device=theta.device)
    m1_c, m2_c, m3_c, m4_c = one_sample_complex_probe_moments(theta, s, laplace_w)

    m1 = torch.real(torch.einsum("ia,a->i", T, m1_c)).to(THEORY_DTYPE)
    m2 = torch.real(torch.einsum("ia,jb,ab->ij", T, T, m2_c)).to(THEORY_DTYPE)
    m3 = torch.real(torch.einsum("ia,jb,kc,abc->ijk", T, T, T, m3_c)).to(THEORY_DTYPE)
    m4 = torch.real(torch.einsum("ia,jb,kc,ld,abcd->ijkl", T, T, T, T, m4_c)).to(THEORY_DTYPE)
    return m1, m2, m3, m4


def sample_mean_raw_moments_from_single(m1, m2, m3, m4, N_obs):
    N = as_real_tensor(N_obs, device=m1.device)
    N2 = N * N
    N3 = N2 * N
    N4 = N2 * N2

    outer11 = torch.einsum("i,j->ij", m1, m1)
    M1 = m1
    M2 = (N * m2 + N * (N - 1.0) * outer11) / N2

    outer111 = torch.einsum("i,j,k->ijk", m1, m1, m1)
    term21 = (
        torch.einsum("ij,k->ijk", m2, m1)
        + torch.einsum("ik,j->ijk", m2, m1)
        + torch.einsum("jk,i->ijk", m2, m1)
    )
    M3 = (
        N * m3
        + N * (N - 1.0) * term21
        + N * (N - 1.0) * (N - 2.0) * outer111
    ) / N3

    outer1111 = torch.einsum("i,j,k,l->ijkl", m1, m1, m1, m1)
    term31 = (
        torch.einsum("ijk,l->ijkl", m3, m1)
        + torch.einsum("ijl,k->ijkl", m3, m1)
        + torch.einsum("ikl,j->ijkl", m3, m1)
        + torch.einsum("jkl,i->ijkl", m3, m1)
    )
    term22 = (
        torch.einsum("ij,kl->ijkl", m2, m2)
        + torch.einsum("ik,jl->ijkl", m2, m2)
        + torch.einsum("il,jk->ijkl", m2, m2)
    )
    term211 = (
        torch.einsum("ij,k,l->ijkl", m2, m1, m1)
        + torch.einsum("ik,j,l->ijkl", m2, m1, m1)
        + torch.einsum("il,j,k->ijkl", m2, m1, m1)
        + torch.einsum("jk,i,l->ijkl", m2, m1, m1)
        + torch.einsum("jl,i,k->ijkl", m2, m1, m1)
        + torch.einsum("kl,i,j->ijkl", m2, m1, m1)
    )
    M4 = (
        N * m4
        + N * (N - 1.0) * (term31 + term22)
        + N * (N - 1.0) * (N - 2.0) * term211
        + N * (N - 1.0) * (N - 2.0) * (N - 3.0) * outer1111
    ) / N4
    return M1, M2, M3, M4


def augment_moments_with_length_feature(m1, m2, m3, m4, length_value):
    D = int(m1.shape[0])
    De = D + 1
    last = D
    c = as_real_tensor(length_value, device=m1.device).reshape(())

    M1 = torch.zeros((De,), dtype=THEORY_DTYPE, device=m1.device)
    M1[:D] = m1
    M1[last] = c

    M2 = torch.zeros((De, De), dtype=THEORY_DTYPE, device=m1.device)
    M2[:D, :D] = m2
    M2[:D, last] = c * m1
    M2[last, :D] = c * m1
    M2[last, last] = c * c

    M3 = torch.zeros((De, De, De), dtype=THEORY_DTYPE, device=m1.device)
    M3[:D, :D, :D] = m3
    M3[last, :D, :D] = c * m2
    M3[:D, last, :D] = c * m2
    M3[:D, :D, last] = c * m2
    M3[last, last, :D] = c * c * m1
    M3[last, :D, last] = c * c * m1
    M3[:D, last, last] = c * c * m1
    M3[last, last, last] = c * c * c

    M4 = torch.zeros((De, De, De, De), dtype=THEORY_DTYPE, device=m1.device)
    M4[:D, :D, :D, :D] = m4
    M4[last, :D, :D, :D] = c * m3
    M4[:D, last, :D, :D] = c * m3
    M4[:D, :D, last, :D] = c * m3
    M4[:D, :D, :D, last] = c * m3
    M4[last, last, :D, :D] = c * c * m2
    M4[last, :D, last, :D] = c * c * m2
    M4[last, :D, :D, last] = c * c * m2
    M4[:D, last, last, :D] = c * c * m2
    M4[:D, last, :D, last] = c * c * m2
    M4[:D, :D, last, last] = c * c * m2
    M4[last, last, last, :D] = c * c * c * m1
    M4[last, last, :D, last] = c * c * c * m1
    M4[last, :D, last, last] = c * c * c * m1
    M4[:D, last, last, last] = c * c * c * m1
    M4[last, last, last, last] = c * c * c * c
    return M1, M2, M3, M4


def feature_moments_from_probe_moments(one_sample_probe_moments, N_obs):
    m1, m2, m3, m4 = one_sample_probe_moments
    M1_probe, M2_probe, M3_probe, M4_probe = sample_mean_raw_moments_from_single(m1, m2, m3, m4, N_obs)
    return augment_moments_with_length_feature(
        M1_probe,
        M2_probe,
        M3_probe,
        M4_probe,
        length_feature_value(N_obs, device=m1.device),
    )


def feature_mean_cov_from_laplace(theta, s, N_obs, laplace_w):
    one_sample = one_sample_real_probe_moments(theta, s, laplace_w)
    M1, M2, _, _ = feature_moments_from_probe_moments(one_sample, N_obs)
    Sx = M2 - torch.einsum("i,j->ij", M1, M1)
    Sx = 0.5 * (Sx + Sx.T)
    return M1, Sx


def feature_mean_only(theta, s, N_obs, laplace_w):
    one_sample = one_sample_real_probe_moments(theta, s, laplace_w)
    M1, _, _, _ = feature_moments_from_probe_moments(one_sample, N_obs)
    return M1




def normalized_w_on_grid(theta, tau_grid, w_func, eps=EPS_W, renorm=True):
    theta = as_real_tensor(theta, device=tau_grid.device)
    tau_grid = as_real_tensor(tau_grid, device=theta.device)
    w = as_real_tensor(w_func(theta, tau_grid), device=theta.device)
    w = torch.clamp(w, min=eps)
    if renorm:
        area = trapz_torch(w, tau_grid)
        w = w / torch.clamp(area, min=eps)
    return w


def make_numeric_laplace_w(w_func, tau_grid, eps=EPS_W, renorm=True):
    tau_grid = as_real_tensor(tau_grid)

    def laplace_w(theta, s):
        theta = as_real_tensor(theta, device=tau_grid.device)
        s = as_complex_tensor(s, device=tau_grid.device)
        w = normalized_w_on_grid(theta, tau_grid, w_func, eps=eps, renorm=renorm)
        integrand = torch.exp(-s * tau_grid.to(dtype=THEORY_COMPLEX_DTYPE)) * w.to(dtype=THEORY_COMPLEX_DTYPE)
        return trapz_torch(integrand, tau_grid)

    return laplace_w


def get_linear_laplace_model_params(model):
    alpha, beta = model.probe_parameters(detach=True)
    s = torch.complex(alpha.to(dtype=THEORY_DTYPE), -beta.to(dtype=THEORY_DTYPE)).to(dtype=THEORY_COMPLEX_DTYPE)
    W, c = model.linearparams()
    return s, W.to(dtype=THEORY_DTYPE), c.to(dtype=THEORY_DTYPE)


def fisher_information_fixedN(
    theta,
    N_obs,
    tau_grid,
    w_func,
    eps_w=EPS_W,
    renorm=True,
    score_func=None,
):
    theta = as_real_tensor(theta)
    tau_grid = as_real_tensor(tau_grid, device=theta.device)
    d = int(theta.numel())

    w0 = normalized_w_on_grid(theta, tau_grid, w_func, eps=eps_w, renorm=renorm)

    if score_func is None and "score_wrt_theta_on_grid" in globals():
        score_func = score_wrt_theta_on_grid

    if score_func is not None:
        candidate = as_real_tensor(
            score_func(theta, tau_grid, w_func=w_func, eps_w=eps_w, renorm=renorm),
            device=theta.device,
        )
        if candidate.shape != (d, tau_grid.numel()):
            raise RuntimeError("score_func must return shape (d, n_grid).")
        if not torch.isfinite(candidate).all():
            raise RuntimeError("score_func returned non-finite values.")
        scores = candidate
    else:
        th_req = theta.detach().clone().requires_grad_(True)

        def _logw(th):
            w = normalized_w_on_grid(th, tau_grid, w_func, eps=eps_w, renorm=renorm)
            return torch.log(torch.clamp(w, min=eps_w))

        jac = torch.autograd.functional.jacobian(_logw, th_req, create_graph=False, strict=False)
        jac = as_real_tensor(jac, device=theta.device)
        if jac.shape != (tau_grid.numel(), d):
            raise RuntimeError("Autograd score Jacobian has unexpected shape.")
        if not torch.isfinite(jac).all():
            raise RuntimeError("Autograd score Jacobian has non-finite values.")
        scores = jac.T

    integrand = w0[:, None, None] * scores.T[:, :, None] * scores.T[:, None, :]
    I1 = trapz_torch(integrand, tau_grid)

    IN = as_real_tensor(N_obs, device=theta.device) * I1
    return 0.5 * (IN + IN.T)


### Exact MSE And biased-CRB Report

这一格把前面的 helper 组装成一个统一接口：

- 给定 `(theta, N_obs)`
- 输出 `eta(theta)`
- bias
- exact covariance / exact MSE
- biased-CRB covariance / MSE



In [ ]:
def linear_laplace_estimator_report(
    model,
    theta,
    N_obs,
    tau_grid,
    w_func,
    laplace_w=None,
    eps_w=EPS_W,
    renorm=True,
    jac_eta_func=None,
    score_func=None,
):
    theta = as_real_tensor(theta)
    d = int(theta.numel())

    s, W, c = get_linear_laplace_model_params(model)

    if laplace_w is None:
        laplace_w = make_numeric_laplace_w(w_func, tau_grid, eps=eps_w, renorm=renorm)

    mx, Sx = feature_mean_cov_from_laplace(theta, s, N_obs, laplace_w)
    eta = W @ mx + c
    Cov_exact = W @ Sx @ W.T
    Cov_exact = 0.5 * (Cov_exact + Cov_exact.T)
    bias = eta - theta
    MSE_exact = Cov_exact + torch.outer(bias, bias)

    if jac_eta_func is None and "jac_eta_wrt_theta" in globals():
        jac_eta_func = jac_eta_wrt_theta

    if jac_eta_func is not None:
        J_eta = as_real_tensor(
            jac_eta_func(
                theta,
                model=model,
                N_obs=N_obs,
                tau_grid=tau_grid,
                w_func=w_func,
                laplace_w=laplace_w,
                eps_w=eps_w,
                renorm=renorm,
            ),
            device=theta.device,
        )
        if J_eta.shape != (d, d):
            raise RuntimeError("jac_eta_func must return shape (d, d).")
        if not torch.isfinite(J_eta).all():
            raise RuntimeError("jac_eta_func returned non-finite values.")
    else:
        th_req = theta.detach().clone().requires_grad_(True)

        def _eta_of_theta(th):
            return estimator_mean_linear_laplace(
                model=model,
                theta=th,
                N_obs=N_obs,
                tau_grid=tau_grid,
                w_func=w_func,
                laplace_w=laplace_w,
                eps_w=eps_w,
                renorm=renorm,
            )

        jac = torch.autograd.functional.jacobian(_eta_of_theta, th_req, create_graph=False, strict=False)
        J_eta = as_real_tensor(jac, device=theta.device)
        if J_eta.shape != (d, d):
            raise RuntimeError("Autograd J_eta has unexpected shape.")
        if not torch.isfinite(J_eta).all():
            raise RuntimeError("Autograd J_eta has non-finite values.")

    I_theta = fisher_information_fixedN(
        theta=theta,
        N_obs=N_obs,
        tau_grid=tau_grid,
        w_func=w_func,
        eps_w=eps_w,
        renorm=renorm,
        score_func=score_func,
    )

    I_inv = torch.linalg.pinv(I_theta)
    Cov_bCRB = J_eta @ I_inv @ J_eta.T
    Cov_bCRB = 0.5 * (Cov_bCRB + Cov_bCRB.T)
    MSE_bCRB = Cov_bCRB + torch.outer(bias, bias)

    return {
        "theta": theta,
        "s": s,
        "W": W,
        "c": c,
        "feature_mean": mx,
        "feature_cov": Sx,
        "eta": eta,
        "bias": bias,
        "J_eta": J_eta,
        "Fisher": I_theta,
        "Cov_exact": Cov_exact,
        "MSE_exact": MSE_exact,
        "Cov_bCRB": Cov_bCRB,
        "MSE_bCRB": MSE_bCRB,
    }

laplace_estimator_report = linear_laplace_estimator_report


### Generalized Barankin Bound

这一节实现的是 test-point Barankin / HCR 风格下界。

它依赖：

- 当前参数点 `theta`
- 一组 test-point offsets `H`
- 以及可选的估计器均值函数 `eta(theta)`

把 bound 做紧，改 `H_test`。


In [ ]:
def estimator_mean_linear_laplace(
    model,
    theta,
    N_obs,
    tau_grid,
    w_func,
    laplace_w=None,
    eps_w=EPS_W,
    renorm=True,
):
    theta = as_real_tensor(theta)
    s, W, c = get_linear_laplace_model_params(model)

    if laplace_w is None:
        laplace_w = make_numeric_laplace_w(w_func, tau_grid, eps=eps_w, renorm=renorm)

    mx, _ = feature_mean_cov_from_laplace(theta, s, N_obs, laplace_w)
    return W @ mx + c


estimator_mean_laplace = estimator_mean_linear_laplace


def generalized_barankin_bound(
    theta,
    H,
    N_obs,
    tau_grid,
    w_func,
    eta_func=None,
    eps_w=EPS_W,
    renorm=True,
):
    theta = as_real_tensor(theta)
    H = as_real_tensor(H, device=theta.device)
    tau_grid = as_real_tensor(tau_grid, device=theta.device)

    d = int(theta.numel())
    M = int(H.shape[0])

    w0 = normalized_w_on_grid(theta, tau_grid, w_func, eps=eps_w, renorm=renorm)

    if HAS_TORCH_FUNC:
        ws = vmap(lambda h: normalized_w_on_grid(theta + h, tau_grid, w_func, eps=eps_w, renorm=renorm))(H)
    else:
        ws = torch.stack(
            [normalized_w_on_grid(theta + H[m], tau_grid, w_func, eps=eps_w, renorm=renorm) for m in range(M)],
            dim=0,
        )
    integrand_A = ((ws[:, None, :] * ws[None, :, :]) / torch.clamp(w0, min=eps_w)).permute(2, 0, 1)
    A = trapz_torch(integrand_A, tau_grid)
    G = torch.expm1(as_real_tensor(N_obs, device=theta.device) * torch.log(torch.clamp(A, min=eps_w)))

    G = 0.5 * (G + G.T)

    if eta_func is None:
        eta0 = theta.clone()
        R = H.T
        bias0 = torch.zeros(d, dtype=THEORY_DTYPE, device=theta.device)
    else:
        eta0 = as_real_tensor(eta_func(theta), device=theta.device)
        R = torch.stack(
            [as_real_tensor(eta_func(theta + H[m]), device=theta.device) - eta0 for m in range(M)],
            dim=1,
        )
        bias0 = eta0 - theta

    G_pinv = torch.linalg.pinv(G)
    Cov_bar = R @ G_pinv @ R.T
    Cov_bar = 0.5 * (Cov_bar + Cov_bar.T)
    MSE_bar = Cov_bar + torch.outer(bias0, bias0)

    return {
        "theta": theta,
        "H": H,
        "G": G,
        "R": R,
        "Cov_barankin": Cov_bar,
        "MSE_barankin": MSE_bar,
        "bias0": bias0,
    }


### Theory Defaults




In [ ]:
tau_grid = torch.linspace(0.0, 20.0, 4000, device=device, dtype=THEORY_DTYPE)

def w_func(theta, tau_grid):
    theta = as_real_tensor(theta, device=tau_grid.device)
    delta = theta[0]
    omega = theta[1]
    return waiting_time_real_torch(tau_grid, delta=delta, omega=omega, eps=EPS_W)

def laplace_w(theta, s):
    theta = as_real_tensor(theta, device=device)
    s = as_complex_tensor(s, device=theta.device)
    return laplace_transform_torch(theta, s, eps=EPS_W)
